In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# =========================================================
# 0) Load + basic cleaning (match your notebook style)
# =========================================================
file_path = "06 - CAS Predictive Modeling Case Competition- Dataset.xlsx"
sheet_name = "4 - Predictive Modeling Case Co"  # 不确定就删掉这一行让它读第一个sheet

df = pd.read_excel(file_path, sheet_name=sheet_name)

# Basic cleanup aligned with your code
df = df.drop_duplicates()
key_cols = ["student_id", "coverage", "claim_id"]
df = df.sort_values(key_cols).drop_duplicates(subset=key_cols, keep="first")

# normalize types
# ===============================
# FIX: BooleanDtype -> int
# ===============================
for col in df.columns:
    if str(df[col].dtype) == "boolean":
        df[col] = df[col].astype(int)

for scol in ["coverage", "class", "study", "greek", "off_campus", "gender", "name"]:
    if scol in df.columns:
        df[scol] = df[scol].astype(str).str.strip()

# cap by coverage limits (same spirit as your code)
coverage_limits = {"Personal Property": 10000.0, "Liability": 500000.0, "Guest Medical": 150000.0}
if "amount" in df.columns:
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    for cov, lim in coverage_limits.items():
        m = (df["coverage"] == cov) & (df["amount"] > lim)
        df.loc[m, "amount"] = lim

df["distance_to_campus"] = pd.to_numeric(df["distance_to_campus"], errors="coerce")
df["log_distance"] = np.log(df["distance_to_campus"].fillna(0) + 1)

# =========================================================
# 1) Build frequency dataset exactly like your approach:
#    has_claim = (claim_id != 0), aggregate to student_id × coverage
# =========================================================
df["has_claim"] = (df["claim_id"] != 0).astype(int)  # :contentReference[oaicite:2]{index=2}

static_features = [
    "class", "study", "gpa", "greek", "off_campus", "distance_to_campus",
    "gender", "sprinklered", "risk_tier", "holdout"
]
static_features = [c for c in static_features if c in df.columns]

freq_df = (
    df.groupby(["student_id", "coverage"], as_index=False)
      .agg(
          claim_count=("has_claim", "sum"),
          **{col: (col, "first") for col in static_features}
      )
)
# add log_distance to freq_df (distance is static per student)
freq_df["log_distance"] = np.log(pd.to_numeric(freq_df["distance_to_campus"], errors="coerce").fillna(0) + 1)

# =========================================================
# 2) Severity dataset: positive amounts only
# =========================================================
sev_df = df.copy()
sev_df["amount"] = pd.to_numeric(sev_df["amount"], errors="coerce")
sev_df = sev_df[sev_df["amount"] > 0].copy()

# =========================================================
# 3) Fit per-coverage models (Poisson freq + Gamma sev)
#    Frequency uses 97.5% cap on TRAIN like your notebook :contentReference[oaicite:3]{index=3}
# =========================================================
FREQ_FORMULA = """
claim_count_capped ~
C(Q("class")) + C(study) + C(greek) + C(off_campus) + C(sprinklered) + C(risk_tier) +
distance_to_campus
"""  # 你 notebook 里 PPP 主公式类似这个（你也做过删 gender/gpa 的合规选择） :contentReference[oaicite:4]{index=4}

SEV_FORMULA = """
amount ~
C(greek) + C(off_campus) + C(sprinklered) + C(risk_tier) +
log_distance
"""

freq_models = {}
sev_models = {}
sev_phi = {}  # dispersion per coverage

coverages = sorted(freq_df["coverage"].dropna().unique())

for cov in coverages:
    # ---------- frequency ----------
    d = freq_df[freq_df["coverage"] == cov].copy()
    d_train = d[d["holdout"] == False].copy()
    d_test  = d[d["holdout"] == True].copy()

    if len(d_train) >= 30:
        cap_value = int(np.nanquantile(d_train["claim_count"], 0.975))
        d_train["claim_count_capped"] = d_train["claim_count"].clip(upper=cap_value)
        d_test["claim_count_capped"]  = d_test["claim_count"].clip(upper=cap_value)

        m = smf.glm(formula=FREQ_FORMULA, data=d_train, family=sm.families.Poisson()).fit()
        freq_models[cov] = (m, cap_value)

    # ---------- severity ----------
    s = sev_df[sev_df["coverage"] == cov].copy()
    s_train = s[s["holdout"] == False].copy()
    if len(s_train) >= 30:
        ms = smf.glm(formula=SEV_FORMULA, data=s_train, family=sm.families.Gamma(sm.families.links.log())).fit()
        sev_models[cov] = ms
        sev_phi[cov] = float(ms.scale)  # statsmodels GLM scale ≈ dispersion φ


# =========================================================
# 4) Monte Carlo: compound Poisson–Gamma
#    shape = 1/phi, scale = mu*phi  (so E[X]=mu, Var[X]=phi*mu^2)
# =========================================================
def simulate_portfolio_total_loss(lam_vec, mu_vec, phi, n_sims=20000, seed=2026):
    rng = np.random.default_rng(seed)
    lam_vec = np.clip(lam_vec, 0, None)
    mu_vec  = np.clip(mu_vec, 1e-12, None)

    shape = 1.0 / phi
    if not np.isfinite(shape) or shape <= 0:
        raise ValueError("phi must be > 0; check severity model scale/dispersion")

    scale_vec = mu_vec * phi

    totals = np.zeros(n_sims)
    for b in range(n_sims):
        N = rng.poisson(lam_vec)  # per policy
        idx = np.where(N > 0)[0]
        if idx.size == 0:
            totals[b] = 0.0
            continue

        counts = N[idx]
        # generate all severities in one shot, then sum back to policies
        sev_draws = rng.gamma(shape=shape, scale=np.repeat(scale_vec[idx], counts))
        # aggregate by policy via bincount
        sums = np.bincount(np.repeat(np.arange(idx.size), counts), weights=sev_draws)
        totals[b] = sums.sum()
    return totals


def summarize_totals(totals):
    q = np.quantile(totals, [0.5, 0.75, 0.9, 0.95, 0.99])
    var95 = np.quantile(totals, 0.95)
    var99 = np.quantile(totals, 0.99)
    tvar95 = totals[totals >= var95].mean() if np.any(totals >= var95) else var95
    tvar99 = totals[totals >= var99].mean() if np.any(totals >= var99) else var99
    return dict(
        mean_total_loss=float(totals.mean()),
        median_total_loss=float(q[0]),
        p75_total_loss=float(q[1]),
        p90_total_loss=float(q[2]),
        p95_total_loss=float(q[3]),
        p99_total_loss=float(q[4]),
        VaR95=float(var95),
        TVaR95=float(tvar95),
        VaR99=float(var99),
        TVaR99=float(tvar99),
    )


# =========================================================
# 5) Run MC by coverage × risk_tier on HOLDOUT (recommended)
# =========================================================
def run_mc_by_coverage_tier(n_sims=20000, loading=0.15, seed=2026):
    rows = []
    for cov in coverages:
        if cov not in freq_models or cov not in sev_models:
            continue

        freq_res, cap_value = freq_models[cov]
        sev_res = sev_models[cov]
        phi = sev_phi[cov]

        # We simulate on holdout population (pricing/validation style)
        d_hold = freq_df[(freq_df["coverage"] == cov) & (freq_df["holdout"] == True)].copy()
        if d_hold.empty:
            continue

        # cap counts same way (not needed for prediction, but keep consistent)
        d_hold["claim_count_capped"] = d_hold["claim_count"].clip(upper=cap_value)

        # lambda_hat from Poisson GLM
        lam_hat = freq_res.predict(d_hold)

        # For mu_hat, we need severity features at policy level.
        # In your data, covariates are static per student; we can use freq_df columns.
        # Build a frame for severity prediction with the columns used in SEV_FORMULA.
        sev_pred_df = d_hold.copy()
        sev_pred_df["log_distance"] = np.log(pd.to_numeric(sev_pred_df["distance_to_campus"], errors="coerce").fillna(0) + 1)

        mu_hat = sev_res.predict(sev_pred_df)

        # group by risk_tier
        for rt, g in d_hold.groupby("risk_tier"):
            idx = g.index
            if len(idx) < 10:
                continue

            totals = simulate_portfolio_total_loss(
                lam_vec=np.asarray(lam_hat.loc[idx], dtype=float),
                mu_vec=np.asarray(mu_hat.loc[idx], dtype=float),
                phi=phi,
                n_sims=n_sims,
                seed=seed
            )

            summ = summarize_totals(totals)

            n_pols = len(idx)
            pure_pp = summ["mean_total_loss"] / n_pols
            gross_pp = pure_pp * (1 + loading)

            rows.append({
                "coverage": cov,
                "risk_tier": rt,
                "n_policies": n_pols,
                "phi_gamma": phi,
                **summ,
                "pure_premium_per_policy": pure_pp,
                "gross_premium_per_policy": gross_pp
            })

    return pd.DataFrame(rows).sort_values(["coverage", "risk_tier"])


mc_results = run_mc_by_coverage_tier(n_sims=20000, loading=0.15, seed=2026)
mc_results

/opt/conda/lib/python3.12/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)


ValueError: The first guess on the deviance function returned a nan.  This could be a boundary  problem and should be reported.